# Module 3 · Convert it into a Single Agent: the Loan Service Agent
⏱ ~15 min

An **agent** is a loop around the model that can:
1. **Understand** the customer request
2. **Choose** between answering directly, checking status, or other tools
3. **Review** the tool result and decide the next step (another tool? respond? escalate?)
4. **Stop** after the task is done (or when a step limit is reached)

```
 user ─► LLM ─► tool call? ──yes──► run approved tool ─► result ─┐
          ▲         │no                                         │
          │         ▼                                           │
          │      final answer ─► STOP                           │
          └─────────────────────────────────────────────────────┘   (max N steps → escalate)
```

In [1]:
import os, sys, json
sys.path.append("..")                      # so we can import utils.py and harness/

from dotenv import load_dotenv
import aisuite as ai
from utils import print_html, print_json, show_messages, show_tool_call, show_trace, show_usage, compare

load_dotenv("../.env", override=True)
MODEL = os.getenv("MODEL", "openai:gpt-4o-mini")   # aisuite format -> "provider:model"
client = ai.Client()                               # reads OPENAI_API_KEY from the environment

print_html({"model": MODEL, "client": "aisuite.Client() ready"}, title="Setup", color="gray")

model,openai:gpt-4o-mini
client,aisuite.Client() ready


## The approved toolset
The functions now live in `harness/tools.py` so the notebook, CLI and Streamlit app all share **one approved list**.

In [2]:
from aisuite.utils.tools import Tools
from harness.tools import check_loan_status, get_missing_documents, get_document_checklist, escalate_to_human

APPROVED_TOOLS = {f.__name__: f for f in
                  [check_loan_status, get_missing_documents, get_document_checklist, escalate_to_human]}
TOOL_SCHEMAS = Tools(list(APPROVED_TOOLS.values())).tools()   # docstrings -> JSON schemas

print_html([{"tool": t["function"]["name"],
             "description": t["function"]["description"].strip().splitlines()[0]} for t in TOOL_SCHEMAS],
           title="Approved tools for the Loan Service Agent", color="amber")

tool,description
check_loan_status,Look up the current status of a loan application.
get_missing_documents,List documents the customer still needs to submit for a loan application.
get_document_checklist,Return the standard list of documents required for a type of loan.
escalate_to_human,Create a ticket so a human loan officer contacts the customer.


In [3]:
AGENT_PROMPT = """You are the Loan Service Agent for Demo Bank.
RULES:
1. Use ONLY your tools for facts about an application. Never guess a status, date or document.
2. You cannot approve, reject, sanction, waive or change any loan. Never claim you did.
3. If no application ID (format LA-1234) is given, ask for it. Never ask for PAN/Aadhaar/account numbers.
4. Call escalate_to_human for complaints, decision-review requests, or anything your tools cannot answer.
5. Be concise and warm. Stop once the request is handled."""

## The agent loop (~30 lines)

In [4]:
def run_agent(user_message: str, max_steps: int = 5):
    messages = [{"role": "system", "content": AGENT_PROMPT},
                {"role": "user", "content": user_message}]
    trace = [{"type": "user", "data": user_message}]

    for step in range(1, max_steps + 1):
        resp = client.chat.completions.create(model=MODEL, messages=messages,
                                              tools=TOOL_SCHEMAS, temperature=0.2)
        msg = resp.choices[0].message
        trace.append({"type": "llm_call", "data": f"step {step}: "
                      + (f"wants {[t.function.name for t in msg.tool_calls]}" if msg.tool_calls else "final answer")})

        if not msg.tool_calls:                                  # DECIDE: done -> respond & STOP
            trace.append({"type": "final", "data": msg.content})
            return msg.content, trace

        messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
            {"id": tc.id, "type": "function",
             "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
            for tc in msg.tool_calls]})

        for tc in msg.tool_calls:                               # ACT: run approved tools only
            name, args = tc.function.name, json.loads(tc.function.arguments or "{}")
            if name not in APPROVED_TOOLS:
                result = {"error": f"{name} is not an approved tool"}
                trace.append({"type": "policy", "data": f"BLOCKED {name}"})
            else:
                result = APPROVED_TOOLS[name](**args)
                trace.append({"type": "tool_call", "data": {"name": name, "args": args}})
                trace.append({"type": "escalation" if name == "escalate_to_human" else "tool_result",
                              "data": result})
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
                                                                # OBSERVE: loop back to the LLM
    ticket = escalate_to_human("UNKNOWN", "Step limit reached")
    trace.append({"type": "escalation", "data": ticket})
    return f"I've passed this to a loan officer (ticket {ticket['ticket_id']}).", trace


def demo(question: str):
    answer, trace = run_agent(question)
    print_html(answer, title=f"🧑 {question}", color="green")
    show_trace(trace, title="What the agent did")

## Scenario 1 · Simple status check → one tool, then respond

In [5]:
demo("Hi, what is the status of my loan application LA-1001?")

## Scenario 2 · Multi-step: the agent *reviews* the result and decides to call a second tool

In [6]:
demo("Why is my personal loan LA-1002 stuck? What do I need to do?")

## Scenario 3 · Complaint → the agent decides to **escalate** to a human

In [7]:
demo("This is ridiculous, LA-1004 was rejected without any reason. I want to complain!")

## Scenario 4 · Request to approve a loan → there is no tool for it

> 🗣️ *Speaker cue:* watch closely. With only a **prompt rule**, the agent often *dodges* (it just repeats the status) instead of clearly saying it cannot approve. Prompts are guidance, not enforcement. The Module 5 harness fixes this with a deterministic decision-request check, a `decision-requests` skill and an output filter.

In [8]:
demo("Please just approve my loan LA-1001, I need the money urgently.")

## Scenario 5 · Missing information → the agent asks instead of guessing

In [9]:
demo("Can you check my loan status?")

## Scenario 6 · Invalid application ID

In [10]:
demo("Status of LA-9999 please")

## ✅ Takeaways
| Agent capability | Where it lives in the code |
|---|---|
| Understand the request | LLM + system prompt |
| Choose: answer directly or use a tool | `msg.tool_calls` present or not |
| Review tool result, decide next step | loop sends results back to the LLM |
| Respond or escalate | `escalate_to_human` tool |
| Stop | no tool calls → return, or `max_steps` reached |
| Only approved actions | `APPROVED_TOOLS` allow-list check |